In [8]:
## 1. Leitura da base de dados
import pandas as pd

df = pd.read_csv("dados/trafego.csv")
df.head()

,Data,Sessões,Usuários ativos,Visualizações
0,1 de jul. de 2023,3109,2047,5566
1,2 de jul. de 2023,3099,2083,5498
2,3 de jul. de 2023,24119,13516,46786
3,4 de jul. de 2023,23635,13335,44984
4,5 de jul. de 2023,22217,13014,43559


In [9]:
## 2. Verificação da estrutura dos dados
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Data             1020 non-null   str  
 1   Sessões          1020 non-null   int64
 2   Usuários ativos  1020 non-null   int64
 3   Visualizações    1020 non-null   int64
dtypes: int64(3), str(1)
memory usage: 32.0 KB


## 3. Conversão da coluna de data

A coluna `Data` foi inicialmente carregada como texto (`str`), e não como data reconhecida pelo Python. Por isso, foi necessário realizar algumas conversões antes de utilizá-la na análise.

Primeiro, os nomes abreviados dos meses em português, como `jul.` e `ago.`, foram substituídos por seus respectivos números (`07`, `08` etc.). Essa etapa foi necessária porque a função de conversão de datas utilizada no pandas trabalha mais facilmente com valores numéricos padronizados no formato dia/mês/ano.

Em seguida, a expressão textual `" de "` foi substituída pelo caractere `/`, transformando datas como `1 de jul. de 2023` em um padrão mais adequado para conversão, como `1/07/2023`.

Por fim, a coluna foi convertida para o tipo `datetime`, que é o formato apropriado para trabalhar com datas no pandas. Essa conversão é importante porque permite extrair variáveis temporais úteis ao modelo, como dia da semana, mês, fim de semana e outras características calendáricas que podem influenciar o comportamento do tráfego.


In [10]:
meses = {
    "jan.": "01", "fev.": "02", "mar.": "03", "abr.": "04",
    "mai.": "05", "jun.": "06", "jul.": "07", "ago.": "08",
    "set.": "09", "out.": "10", "nov.": "11", "dez.": "12"
}

for mes_pt, mes_num in meses.items():
    df["Data"] = df["Data"].str.replace(mes_pt, mes_num, regex=False)

df["Data"] = df["Data"].str.replace(" de ", "/", regex=False)
df["Data"] = pd.to_datetime(df["Data"], format="%d/%m/%Y")

df.head()

,Data,Sessões,Usuários ativos,Visualizações
0,2023-07-01,3109,2047,5566
1,2023-07-02,3099,2083,5498
2,2023-07-03,24119,13516,46786
3,2023-07-04,23635,13335,44984
4,2023-07-05,22217,13014,43559


## 4. Criação da variável dia da semana

Nesta etapa, é criada uma variável representando o dia da semana de cada observação. Essa informação pode ser relevante para o modelo, pois o tráfego do portal apresenta comportamento distinto entre dias úteis e finais de semana.

In [12]:
df["dia_semana"] = df["Data"].dt.dayofweek + 1
df[["Data", "dia_semana"]].head(10)

,Data,dia_semana
0,2023-07-01,6
1,2023-07-02,7
2,2023-07-03,1
3,2023-07-04,2
4,2023-07-05,3
5,2023-07-06,4
6,2023-07-07,5
7,2023-07-08,6
8,2023-07-09,7
9,2023-07-10,1


## 5. Criação da variável indicadora de fim de semana

Além do dia da semana, foi criada uma variável binária para identificar se a observação pertence a um fim de semana. Essa variável pode ser útil porque o tráfego do portal tende a cair significativamente aos sábados e domingos.

In [13]:
df["fim_de_semana"] = (df["Data"].dt.dayofweek >= 5).astype(int)
df[["Data", "dia_semana", "fim_de_semana"]].head(10)

,Data,dia_semana,fim_de_semana
0,2023-07-01,6,1
1,2023-07-02,7,1
2,2023-07-03,1,0
3,2023-07-04,2,0
4,2023-07-05,3,0
5,2023-07-06,4,0
6,2023-07-07,5,0
7,2023-07-08,6,1
8,2023-07-09,7,1
9,2023-07-10,1,0


## 6. Criação da variável mês

Também foi criada uma variável correspondente ao mês de cada observação. Essa informação pode ajudar o modelo a captar padrões sazonais mais amplos no comportamento do tráfego ao longo do ano.

In [16]:
df["mes"] = df["Data"].dt.month
df[["Data", "mes"]].head(90)

,Data,mes
0,2023-07-01,7
1,2023-07-02,7
2,2023-07-03,7
3,2023-07-04,7
4,2023-07-05,7
...,...,...
85,2023-09-24,9
86,2023-09-25,9
87,2023-09-26,9
88,2023-09-27,9


## 8. Criação da variável de recesso judiciário

Foi criada uma variável binária para identificar o período de recesso judiciário, compreendido entre 20 de dezembro e 6 de janeiro. Essa variável foi incluída porque esse intervalo tende a alterar de forma relevante o padrão de utilização do portal.

In [19]:
df["recesso_judiciario"] = (
    ((df["Data"].dt.month == 12) & (df["Data"].dt.day >= 20)) |
    ((df["Data"].dt.month == 1) & (df["Data"].dt.day <= 6))
).astype(int)



## 9. Criação da variável de feriados nacionais de data fixa

Optou-se por considerar apenas os feriados nacionais com data fixa. Essa escolha torna a variável mais simples de reproduzir em previsões futuras, sem necessidade de calcular datas móveis, como Paixão de Cristo. 

In [22]:
feriados_fixos = [
    (1, 1),    # Confraternização Universal
    (4, 21),   # Tiradentes
    (5, 1),    # Dia do Trabalho
    (9, 7),    # Independência do Brasil
    (10, 12),  # Nossa Senhora Aparecida
    (11, 2),   # Finados
    (11, 15),  # Proclamação da República
    (12, 25)   # Natal
]

datas_feriados = []

ano_inicial = df["Data"].dt.year.min()
ano_final = df["Data"].dt.year.max()

for ano in range(ano_inicial, ano_final + 1):
    for mes, dia in feriados_fixos:
        datas_feriados.append(pd.Timestamp(year=ano, month=mes, day=dia))
    
    # Consciência Negra: só entra a partir de 2024
    if ano >= 2024:
        datas_feriados.append(pd.Timestamp(year=ano, month=11, day=20))

feriados_nacionais_fixos = pd.DatetimeIndex(datas_feriados)

# Mantém apenas as datas que estão dentro do intervalo da sua base
feriados_nacionais_fixos = feriados_nacionais_fixos[
    (feriados_nacionais_fixos >= df["Data"].min()) &
    (feriados_nacionais_fixos <= df["Data"].max())
]

df["feriado_nacional_fixo"] = df["Data"].isin(feriados_nacionais_fixos).astype(int)

df.loc[df["feriado_nacional_fixo"] == 1, ["Data", "feriado_nacional_fixo"]]

,Data,feriado_nacional_fixo
68,2023-09-07,1
103,2023-10-12,1
124,2023-11-02,1
137,2023-11-15,1
177,2023-12-25,1
184,2024-01-01,1
295,2024-04-21,1
305,2024-05-01,1
434,2024-09-07,1
469,2024-10-12,1


In [24]:
df.loc[df["feriado_nacional_fixo"] == 1, "Data"].dt.year.value_counts().sort_index()

Data
2023    5
2024    9
2025    9
2026    1
Name: count, dtype: int64

Salvamento da base tratada até aqui

In [25]:
df.to_csv("dados/trafego_tratado.csv", index=False)